# 🏥 Medical Knowledge Graph: Ingestion Pipeline (Kaggle Optimized)

### 🚀 Optimization: Dual T4 GPU + RAM-Efficient Loading
1. **Pre-Download:** We download the model weights once to avoid concurrent lock issues.
2. **Parallel Workers:** Two independent processes, each pinned to a separate T4 GPU.
3. **RAM-Safe:** Uses `low_cpu_mem_usage` to fit the 8B model into Kaggle's 13GB RAM.

In [ ]:
# 1. Environment Setup
!git clone https://github.com/marmor123/medical-knowledge-graph.git
%cd medical-knowledge-graph
!pip install -r requirements.txt
!pip install bitsandbytes>=0.46.1 json-repair

# 2. Pre-download model to avoid parallel lock issues
from src.etl.vlm_parser import VLMParser
print("📥 Pre-downloading model weights...")
dummy = VLMParser()
print("✅ Weights cached.")

## 🧬 Parallel Ingestion (Pages 1-416)
Splitting the work across both T4 GPUs.

In [ ]:
import subprocess
import time
import os

pdf_file = 'Pocket Medicine 9th Edition 2026.pdf'

# GPU 0 Job
cmd0 = f"CUDA_VISIBLE_DEVICES=0 python -m src.etl.processor --pdf '{pdf_file}' --limit 208 --out 'data/interim/raw_chunks_gpu0.jsonl'"

# GPU 1 Job
cmd1 = f"CUDA_VISIBLE_DEVICES=1 python -m src.etl.processor --pdf '{pdf_file}' --start 209 --limit 208 --out 'data/interim/raw_chunks_gpu1.jsonl'"

print("🚀 Launching Dual-GPU Workers...")
p0 = subprocess.Popen(cmd0, shell=True)
p1 = subprocess.Popen(cmd1, shell=True)

while p0.poll() is None or p1.poll() is None:
    print(f"Ingestion active... [{time.strftime('%H:%M:%S')}] Check files in data/interim/")
    time.sleep(120)

print("✅ Ingestion Complete! Merging outputs...")
!cat data/interim/raw_chunks_gpu0.jsonl data/interim/raw_chunks_gpu1.jsonl > data/interim/raw_chunks.jsonl

In [ ]:
# Upload results to GitHub
from src.etl.processor import log_error # Just to ensure path is right
# [Insert your upload_extracted_data logic here]